In [ ]:
# 1. IMPORT REQUIRED LIBRARIES
import feedparser
import newspaper
import requests
import csv
import hashlib
import json
import xml.etree.ElementTree as ET
import re
import logging
import sys
import time
from datetime import datetime, timedelta
from urllib.parse import urljoin, urlparse

# Import data standardizer
sys.path.insert(0, '/root/workspace')
from data_standardizer import standardize_dataset

print("✅ Libraries imported")

✅ Libraries imported


In [49]:
# 2. CONFIGURE LOGGING
LOG_FILENAME = "scraper_execution.log"
LOG_FORMAT = "%(asctime)s [%(levelname)-8s] %(message)s"
LOG_DATEFMT = "%Y-%m-%d %H:%M:%S"

logger = logging.getLogger("NewsScraperModule")
logger.setLevel(logging.DEBUG)

# File handler
file_handler = logging.FileHandler(LOG_FILENAME, encoding='utf-8')
file_handler.setLevel(logging.DEBUG)
file_formatter = logging.Formatter(LOG_FORMAT, datefmt=LOG_DATEFMT)
file_handler.setFormatter(file_formatter)

# Console handler
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_formatter = logging.Formatter("[%(levelname)s] %(message)s")
console_handler.setFormatter(console_formatter)

logger.addHandler(file_handler)
logger.addHandler(console_handler)

logger.info("=" * 70)
logger.info("NEWS SCRAPER MODULE STARTED")
logger.info("=" * 70)
print("✅ Logging configured")

[INFO] ======================================================================
[INFO] ======================================================================
[INFO] ======================================================================
[INFO] ======================================================================
[INFO] ======================================================================
[INFO] NEWS SCRAPER MODULE STARTED
[INFO] NEWS SCRAPER MODULE STARTED
[INFO] NEWS SCRAPER MODULE STARTED
[INFO] NEWS SCRAPER MODULE STARTED
[INFO] NEWS SCRAPER MODULE STARTED
[INFO] ======================================================================
[INFO] ======================================================================
[INFO] ======================================================================
[INFO] ======================================================================
[INFO] ======================================================================
[INFO] ======================================

✅ Logging configured


In [50]:
# 3. CONFIGURATION

OUTPUT_FILENAME = "scraped_data.csv"
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'

# Scraper settings
DAYS_LOOKBACK = 14
MIN_CONTENT_LENGTH = 300
REQUEST_TIMEOUT = 5
MAX_ARTICLES_PER_SOURCE = 25

# News sources - CATEGORIZED BY DIFFICULTY LEVEL & TYPE
# ====================================================================
SOURCES_CONFIG = [
    # EASY SOURCES (Reliable RSS, well-structured, English)
    # Tech News - Proven RSS feeds
    {"domain_name": "TechCrunch", "base_url": "https://techcrunch.com", "difficulty": "easy", "type": "tech", "language": "English"},
    {"domain_name": "The Verge", "base_url": "https://www.theverge.com", "difficulty": "easy", "type": "tech", "language": "English"},
    {"domain_name": "Ars Technica", "base_url": "https://arstechnica.com", "difficulty": "easy", "type": "tech", "language": "English"},
    {"domain_name": "Engadget", "base_url": "https://www.engadget.com", "difficulty": "easy", "type": "tech", "language": "English"},
    
    # Business/General News - Strong RSS
    {"domain_name": "Reuters", "base_url": "https://www.reuters.com", "difficulty": "easy", "type": "general", "language": "English"},
    {"domain_name": "AP News", "base_url": "https://apnews.com", "difficulty": "easy", "type": "general", "language": "English"},
    {"domain_name": "The Guardian", "base_url": "https://www.theguardian.com", "difficulty": "easy", "type": "general", "language": "English"},
    
    # MEDIUM SOURCES (RSS available but may be inconsistent, or require HTML fallback)
    # Tech/Business hybrids
    {"domain_name": "Wired", "base_url": "https://www.wired.com", "difficulty": "medium", "type": "tech", "language": "English"},
    {"domain_name": "MIT Technology Review", "base_url": "https://www.technologyreview.com", "difficulty": "medium", "type": "tech", "language": "English"},
    {"domain_name": "VentureBeat", "base_url": "https://venturebeat.com", "difficulty": "medium", "type": "business", "language": "English"},
    {"domain_name": "Protocol", "base_url": "https://protocol.com", "difficulty": "medium", "type": "tech", "language": "English"},
    {"domain_name": "The Markup", "base_url": "https://themarkup.org", "difficulty": "medium", "type": "tech", "language": "English"},
    
    # Business Focus
    {"domain_name": "Bloomberg", "base_url": "https://www.bloomberg.com", "difficulty": "medium", "type": "business", "language": "English"},
    {"domain_name": "Quartz", "base_url": "https://qz.com", "difficulty": "medium", "type": "business", "language": "English"},
    
    # International perspectives
    {"domain_name": "Al Jazeera", "base_url": "https://www.aljazeera.com", "difficulty": "medium", "type": "general", "language": "English"},
    {"domain_name": "DW News", "base_url": "https://www.dw.com", "difficulty": "medium", "type": "general", "language": "English"},
    
    # HARD SOURCES (Limited/inconsistent RSS, heavy HTML fallback needed, slow)
    # Popular but challenging to scrape
    {"domain_name": "BBC", "base_url": "https://www.bbc.com", "difficulty": "hard", "type": "general", "language": "English"},
    {"domain_name": "CNBC", "base_url": "https://www.cnbc.com", "difficulty": "hard", "type": "business", "language": "English"},
    {"domain_name": "Financial Times", "base_url": "https://www.ft.com", "difficulty": "hard", "type": "business", "language": "English"},
    {"domain_name": "Mashable", "base_url": "https://mashable.com", "difficulty": "hard", "type": "tech", "language": "English"},
    
    # THAI SOURCES (Mixed difficulty - testing multi-language support)
    # Easy Thai
    {"domain_name": "Blognone", "base_url": "https://www.blognone.com", "difficulty": "easy", "type": "tech", "language": "Thai"},
    {"domain_name": "BearTai", "base_url": "https://www.beartai.com", "difficulty": "medium", "type": "tech", "language": "Thai"},
    
    # Medium Thai
    {"domain_name": "Thai PBS", "base_url": "https://www.thaipbs.or.th", "difficulty": "medium", "type": "general", "language": "Thai"},
    {"domain_name": "Bangkok Post", "base_url": "https://www.bangkokpost.com", "difficulty": "medium", "type": "general", "language": "English"},
    
    # Hard Thai
    {"domain_name": "Matichon", "base_url": "https://www.matichon.co.th", "difficulty": "hard", "type": "general", "language": "Thai"},
]

# Keywords for filtering (Multi-language support)
KEYWORDS_CONFIG = [
    # English keywords
    "AI", "Artificial Intelligence", "Machine Learning", "Data", 
    "Google", "Microsoft", "Meta", "NVIDIA", "Crypto", "Bitcoin",
    "Technology", "Innovation", "Algorithm", "Neural", "Model",
    "OpenAI", "ChatGPT", "Gemini", "Quantum", "Robot",
    
    # Thai keywords (for Thai language sources)
    "เทคโนโลยี", "ไอที", "ปัญญาประดิษฐ์", "AI", "ข้อมูล",
    "กูเกิล", "ไมโครซอฟท์", "เน็ตเวิร์ก", "ดิจิทัล", "ออนไลน์"
]

logger.info(f"Configuration: {len(SOURCES_CONFIG)} sources across:")
logger.info(f"  Easy: {sum(1 for s in SOURCES_CONFIG if s['difficulty'] == 'easy')}")
logger.info(f"  Medium: {sum(1 for s in SOURCES_CONFIG if s['difficulty'] == 'medium')}")
logger.info(f"  Hard: {sum(1 for s in SOURCES_CONFIG if s['difficulty'] == 'hard')}")
logger.info(f"  Languages: English, Thai")
logger.info(f"  Topics: Tech, Business, General News")

print(f"✅ Config ready: {len(SOURCES_CONFIG)} sources")
print(f"   📊 Easy: {sum(1 for s in SOURCES_CONFIG if s['difficulty'] == 'easy')}, "
      f"Medium: {sum(1 for s in SOURCES_CONFIG if s['difficulty'] == 'medium')}, "
      f"Hard: {sum(1 for s in SOURCES_CONFIG if s['difficulty'] == 'hard')}")
print(f"   🌍 Languages: English (17), Thai (4)")
print(f"   📚 Topics: Tech, Business, General News")
print(f"   🔑 Keywords: {len(KEYWORDS_CONFIG)} total ({sum(1 for k in KEYWORDS_CONFIG if len(k) < 10)} English, {sum(1 for k in KEYWORDS_CONFIG if len(k) >= 10)} Thai)")


[INFO] Configuration: 25 sources across:
[INFO] Configuration: 25 sources across:
[INFO] Configuration: 25 sources across:
[INFO] Configuration: 25 sources across:
[INFO] Configuration: 25 sources across:
[INFO]   Easy: 8
[INFO]   Easy: 8
[INFO]   Easy: 8
[INFO]   Easy: 8
[INFO]   Easy: 8
[INFO]   Medium: 12
[INFO]   Medium: 12
[INFO]   Medium: 12
[INFO]   Medium: 12
[INFO]   Medium: 12
[INFO]   Hard: 5
[INFO]   Hard: 5
[INFO]   Hard: 5
[INFO]   Hard: 5
[INFO]   Hard: 5
[INFO] Configuration: 25 sources across:
[INFO] Configuration: 25 sources across:
[INFO] Configuration: 25 sources across:
[INFO] Configuration: 25 sources across:
[INFO]   Easy: 8
[INFO]   Easy: 8
[INFO]   Easy: 8
[INFO]   Easy: 8
[INFO]   Easy: 8
[INFO]   Medium: 12
[INFO]   Medium: 12
[INFO]   Medium: 12
[INFO]   Medium: 12
[INFO]   Medium: 12
[INFO]   Hard: 5
[INFO]   Hard: 5
[INFO]   Hard: 5
[INFO]   Hard: 5
[INFO]   Hard: 5
[INFO]   Languages: English, Thai
[INFO]   Languages: English, Thai
[INFO]   Languages: Eng

✅ Config ready: 25 sources
   📊 Easy: 8, Medium: 12, Hard: 5
   🌍 Languages: English (17), Thai (4)
   📚 Topics: Tech, Business, General News
   🔑 Keywords: 30 total (23 English, 7 Thai)


In [51]:
# 4. HELPER FUNCTIONS

def generate_hash(url):
    """Generate SHA-256 hash for URL deduplication"""
    return hashlib.sha256(url.encode()).hexdigest()

def is_date_valid(date_tuple):
    """Check if date is within lookback period"""
    if not date_tuple:
        return True
    try:
        published_date = datetime(*date_tuple[:6])
        cutoff_date = datetime.now() - timedelta(days=DAYS_LOOKBACK)
        return published_date >= cutoff_date
    except:
        return True

def get_matched_keywords(headline, content, keywords):
    """Find matching keywords (case-insensitive)"""
    text = (headline + " " + content).lower()
    matched = []
    for keyword in keywords:
        if keyword.lower() in text:
            matched.append(keyword)
    return matched

def strip_html_tags(html_text):
    """Remove HTML tags from text"""
    if not html_text:
        return ""
    html_text = re.sub(r'<script[^>]*>.*?</script>', '', html_text, flags=re.DOTALL)
    html_text = re.sub(r'<style[^>]*>.*?</style>', '', html_text, flags=re.DOTALL)
    html_text = re.sub(r'<[^>]+>', '', html_text)
    return html_text

def safe_request(url, timeout=REQUEST_TIMEOUT):
    """Make HTTP request safely"""
    try:
        headers = {'User-Agent': USER_AGENT}
        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        return response
    except Exception as e:
        logger.debug(f"Request failed: {str(e)[:40]}")
        return None

logger.info("Helper functions loaded")
print("✅ Helper functions ready")

[INFO] Helper functions loaded
[INFO] Helper functions loaded
[INFO] Helper functions loaded
[INFO] Helper functions loaded
[INFO] Helper functions loaded
[INFO] Helper functions loaded
[INFO] Helper functions loaded
[INFO] Helper functions loaded
[INFO] Helper functions loaded


✅ Helper functions ready


In [52]:
# 5. RSS SCRAPER

def resolve_rss_link(base_url):
    """Find RSS feed URL"""
    patterns = ["/feed", "/feeds/latest.xml", "/rss", "/rss.xml", "/feed/rss", "/atom.xml", "/feed.xml"]
    
    for pattern in patterns:
        rss_url = urljoin(base_url, pattern)
        try:
            feed = feedparser.parse(rss_url)
            if feed.entries:
                return rss_url
        except:
            continue
    return None

def scrape_rss(sources, keywords):
    """Scrape articles from RSS feeds"""
    all_articles = []
    logger.info("Starting RSS scraping...")
    
    for source in sources:
        logger.info(f"Processing: {source['domain_name']}")
        
        feed_url = resolve_rss_link(source['base_url'])
        if not feed_url:
            logger.debug(f"  No RSS feed found")
            continue
        
        try:
            feed = feedparser.parse(feed_url)
            
            for entry in feed.entries[:MAX_ARTICLES_PER_SOURCE]:
                try:
                    headline = entry.get('title', 'Unknown')
                    article_url = entry.get('link', '')
                    
                    if not article_url:
                        continue
                    
                    if hasattr(entry, 'published_parsed') and not is_date_valid(entry.published_parsed):
                        continue
                    
                    # Download article
                    article = newspaper.Article(article_url)
                    article.download()
                    article.parse()
                    
                    if not article.text or len(article.text) < MIN_CONTENT_LENGTH:
                        continue
                    
                    matched = get_matched_keywords(headline, article.text, keywords)
                    if not matched:
                        continue
                    
                    row = {
                        "source": source['domain_name'],
                        "headline": headline,
                        "author": article.authors[0] if article.authors else "Unknown",
                        "url": article_url,
                        "published": entry.get('published', datetime.now().isoformat()),
                        "keywords": matched,
                        "content_snippet": article.text[:100],
                        "url_hash": generate_hash(article_url),
                        "full_content": article.text,
                        "matched_tags": matched,
                        "status": "New",
                        "method": "rss"
                    }
                    all_articles.append(row)
                    logger.debug(f"  ✓ {headline[:40]}...")
                    
                except Exception as e:
                    logger.debug(f"  Error: {str(e)[:30]}")
                    continue
                    
        except Exception as e:
            logger.debug(f"  RSS error: {str(e)[:30]}")
            continue
    
    logger.info(f"RSS complete: {len(all_articles)} articles")
    return all_articles

print("✅ RSS scraper ready")

✅ RSS scraper ready


In [53]:
# 6. STATIC HTML SCRAPER

def discover_sitemaps(base_url):
    """Find sitemap URLs from robots.txt"""
    sitemaps = []
    try:
        robots_url = urljoin(base_url, "/robots.txt")
        response = safe_request(robots_url)
        if response:
            for line in response.text.split('\n'):
                if line.lower().startswith('sitemap:'):
                    sitemap_url = line.split(':', 1)[1].strip()
                    sitemaps.append(sitemap_url)
    except:
        pass
    return sitemaps

def parse_sitemap_urls(sitemap_url):
    """Extract URLs from XML sitemap"""
    urls = []
    try:
        response = safe_request(sitemap_url)
        if response:
            root = ET.fromstring(response.content)
            for loc in root.findall('.//{http://www.sitemaps.org/schemas/sitemap/0.9}loc'):
                urls.append(loc.text)
    except:
        pass
    return urls

def scrape_html(sources, keywords):
    """Scrape articles from static HTML"""
    all_articles = []
    logger.info("Starting HTML scraping...")
    
    for source in sources:
        logger.info(f"Processing: {source['domain_name']}")
        article_count = 0
        
        # Get URLs from sitemaps
        article_urls = []
        sitemaps = discover_sitemaps(source['base_url'])
        
        for sitemap_url in sitemaps:
            urls = parse_sitemap_urls(sitemap_url)
            article_urls.extend(urls)
        
        logger.debug(f"  Found {len(article_urls)} URLs")
        
        # Process articles
        for url in article_urls[:MAX_ARTICLES_PER_SOURCE]:
            if article_count >= MAX_ARTICLES_PER_SOURCE:
                break
                
            try:
                article = newspaper.Article(url)
                article.download()
                article.parse()
                
                if not article.text or len(article.text) < MIN_CONTENT_LENGTH:
                    continue
                
                matched = get_matched_keywords(article.title, article.text, keywords)
                if not matched:
                    continue
                
                row = {
                    "source": source['domain_name'],
                    "headline": article.title or "Unknown",
                    "author": article.authors[0] if article.authors else "Unknown",
                    "url": url,
                    "published": article.publish_date.isoformat() if article.publish_date else datetime.now().isoformat(),
                    "keywords": matched,
                    "content_snippet": article.text[:100],
                    "url_hash": generate_hash(url),
                    "full_content": article.text,
                    "matched_tags": matched,
                    "status": "New",
                    "method": "html"
                }
                all_articles.append(row)
                article_count += 1
                logger.debug(f"  ✓ {article.title[:40] if article.title else 'Unknown'}...")
                
            except Exception as e:
                logger.debug(f"  Error: {str(e)[:30]}")
                continue
        
        logger.debug(f"  Extracted {article_count} articles")
    
    logger.info(f"HTML complete: {len(all_articles)} articles")
    return all_articles

print("✅ HTML scraper ready")

✅ HTML scraper ready


In [54]:
# 7. API SCRAPER

KNOWN_API_PATTERNS = {
    "wordpress": {
        "endpoint": "/wp-json/wp/v2/posts",
        "params": {"per_page": 100, "orderby": "date"},
        "type": "wordpress"
    }
}

def discover_api(base_url, domain):
    """Discover API endpoints"""
    for cms, pattern_info in KNOWN_API_PATTERNS.items():
        api_url = urljoin(base_url, pattern_info["endpoint"])
        response = safe_request(api_url)
        if response and response.status_code == 200:
            try:
                response.json()
                logger.debug(f"  Found {cms} API")
                return {"endpoint": pattern_info["endpoint"], "type": cms}
            except:
                pass
    return None

def parse_wordpress_api(json_data):
    """Parse WordPress API response"""
    articles = []
    if isinstance(json_data, list):
        for item in json_data:
            articles.append({
                "headline": item.get("title", {}).get("rendered", "Unknown"),
                "content": item.get("content", {}).get("rendered", ""),
                "author": item.get("_embedded", {}).get("author", [{}])[0].get("name", "Unknown"),
                "url": item.get("link", ""),
                "published": item.get("date", datetime.now().isoformat())
            })
    return articles

def scrape_api(sources, keywords):
    """Scrape articles from APIs"""
    all_articles = []
    logger.info("Starting API scraping...")
    
    for source in sources:
        logger.info(f"Processing: {source['domain_name']}")
        
        api_info = discover_api(source['base_url'], source['domain_name'])
        if not api_info:
            logger.debug(f"  No API found")
            continue
        
        try:
            api_url = urljoin(source['base_url'], api_info["endpoint"])
            params = KNOWN_API_PATTERNS.get(api_info["type"], {}).get("params", {})
            
            response = safe_request(api_url)
            if not response:
                continue
            
            json_data = response.json()
            
            if api_info["type"] == "wordpress":
                articles = parse_wordpress_api(json_data)
            else:
                articles = []
            
            logger.debug(f"  Parsed {len(articles)} articles")
            
            for article in articles[:MAX_ARTICLES_PER_SOURCE]:
                try:
                    content = strip_html_tags(article.get("content", ""))
                    headline = article.get("headline", "")
                    url = article.get("url", "")
                    
                    if not url or len(content) < MIN_CONTENT_LENGTH:
                        continue
                    
                    matched = get_matched_keywords(headline, content, keywords)
                    if not matched:
                        continue
                    
                    row = {
                        "source": source['domain_name'],
                        "headline": headline,
                        "author": article.get("author", "Unknown"),
                        "url": url,
                        "published": article.get("published", datetime.now().isoformat()),
                        "keywords": matched,
                        "content_snippet": content[:100],
                        "url_hash": generate_hash(url),
                        "full_content": content,
                        "matched_tags": matched,
                        "status": "New",
                        "method": "api"
                    }
                    all_articles.append(row)
                    logger.debug(f"  ✓ {headline[:40]}...")
                    
                except Exception as e:
                    logger.debug(f"  Error: {str(e)[:30]}")
                    continue
                    
        except Exception as e:
            logger.debug(f"  API error: {str(e)[:30]}")
            continue
    
    logger.info(f"API complete: {len(all_articles)} articles")
    return all_articles

print("✅ API scraper ready")

✅ API scraper ready


In [55]:
# 8. DATA CLEANING & EXPORT

def clean_and_standardize(articles):
    """Clean, standardize, and deduplicate articles"""
    logger.info(f"Cleaning {len(articles)} articles...")
    
    # Standardize all fields
    articles = standardize_dataset(articles)
    
    # Remove duplicates by url_hash
    seen_urls = set()
    unique_articles = []
    duplicates = 0
    
    for article in articles:
        url_hash = article.get("url_hash", "")
        if url_hash and url_hash not in seen_urls:
            seen_urls.add(url_hash)
            unique_articles.append(article)
        else:
            duplicates += 1
    
    logger.info(f"Clean complete: {len(unique_articles)} unique, {duplicates} duplicates removed")
    return unique_articles

def export_to_csv(articles, filename):
    """Export articles to CSV"""
    logger.info(f"Exporting to {filename}...")
    
    if not articles:
        logger.warning("No articles to export")
        return 0
    
    try:
        fieldnames = articles[0].keys()
        
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(articles)
        
        logger.info(f"✅ Exported {len(articles)} articles to {filename}")
        return len(articles)
        
    except Exception as e:
        logger.error(f"Export error: {str(e)}")
        return 0

print("✅ Export function ready")

✅ Export function ready


In [56]:
# 9. EXECUTE SCRAPER - OPTIMIZED FALLBACK (RSS → HTML with timeouts)

# Problematic sources that are too slow - skip them or limit to RSS only
SKIP_HTML_SOURCES = {"BBC", "CNBC", "Financial Times"}  # These hang on HTML scraping
MAX_HTML_TIME = 30  # Max 30 seconds for HTML fallback per source

def scrape_with_smart_fallback(source, keywords):
    """
    Try RSS first (fast). If no RSS or 0 articles, fallback to HTML scraping.
    - RSS: Always try (fast)
    - HTML: Only for whitelisted sources, with timeout
    - Skips slow/problematic sources
    """
    source_name = source['domain_name']
    base_url = source['base_url']
    articles = []
    
    # STEP 1: Try RSS (FAST - preferred method)
    try:
        feed_url = resolve_rss_link(base_url)
        if feed_url:
            feed = feedparser.parse(feed_url)
            logger.debug(f"{source_name}: Found RSS with {len(feed.entries)} entries")
            
            for entry in feed.entries[:MAX_ARTICLES_PER_SOURCE]:
                try:
                    headline = entry.get('title', 'Unknown')
                    article_url = entry.get('link', '')
                    
                    if not article_url:
                        continue
                    
                    # Filter by date
                    if hasattr(entry, 'published_parsed') and not is_date_valid(entry.published_parsed):
                        continue
                    
                    # Download and parse article
                    article = newspaper.Article(article_url)
                    article.download()
                    article.parse()
                    
                    # Filter by content length
                    if not article.text or len(article.text) < MIN_CONTENT_LENGTH:
                        continue
                    
                    # Filter by keywords
                    matched = get_matched_keywords(headline, article.text, keywords)
                    if not matched:
                        continue
                    
                    # Create article record
                    articles.append({
                        "source": source_name,
                        "headline": headline,
                        "author": article.authors[0] if article.authors else "Unknown",
                        "url": article_url,
                        "published": entry.get('published', datetime.now().isoformat()),
                        "keywords": matched,
                        "content_snippet": article.text[:100],
                        "url_hash": generate_hash(article_url),
                        "full_content": article.text,
                        "matched_tags": matched,
                        "status": "New",
                        "method": "rss"
                    })
                    logger.debug(f"  ✓ RSS: {headline[:40]}...")
                    
                except Exception as e:
                    logger.debug(f"  RSS entry error: {str(e)[:30]}")
                    continue
            
            # If RSS found articles, return them (success!)
            if articles:
                return articles, "rss"
    
    except Exception as e:
        logger.debug(f"{source_name}: RSS error - {str(e)[:40]}")
    
    # STEP 2: HTML Fallback (SLOWER - for sites without RSS)
    # Skip problematic sources to save time
    if source_name in SKIP_HTML_SOURCES:
        logger.debug(f"{source_name}: Skipping HTML (known to be slow)")
        return [], "skip"
    
    try:
        html_start = time.time()
        logger.debug(f"{source_name}: Trying HTML fallback...")
        
        # Strategy 2a: Try sitemaps first (faster)
        sitemaps = discover_sitemaps(base_url)
        article_urls = []
        
        for sitemap_url in sitemaps[:2]:  # Limit to 2 sitemaps (speed)
            urls = parse_sitemap_urls(sitemap_url)
            article_urls.extend(urls[:15])  # Max 15 URLs per sitemap
        
        # Strategy 2b: If no sitemaps, try homepage scraping (newspaper build)
        if not article_urls:
            logger.debug(f"{source_name}: No sitemaps, trying homepage...")
            response = safe_request(base_url)
            if response:
                try:
                    from newspaper import build
                    paper = build(base_url, memoize_articles=False)
                    article_urls = [a.url for a in paper.articles[:10]]  # Max 10 from homepage
                except:
                    pass
        
        # Process discovered URLs (with timeout check)
        for url in article_urls[:MAX_ARTICLES_PER_SOURCE]:
            # Check if we've exceeded time limit
            elapsed = time.time() - html_start
            if elapsed > MAX_HTML_TIME:
                logger.debug(f"{source_name}: HTML timeout ({elapsed:.0f}s), stopping")
                break
            
            try:
                article = newspaper.Article(url)
                article.download()
                article.parse()
                
                if not article.text or len(article.text) < MIN_CONTENT_LENGTH:
                    continue
                
                matched = get_matched_keywords(article.title or "", article.text, keywords)
                if not matched:
                    continue
                
                articles.append({
                    "source": source_name,
                    "headline": article.title or "Unknown",
                    "author": article.authors[0] if article.authors else "Unknown",
                    "url": url,
                    "published": article.publish_date.isoformat() if article.publish_date else datetime.now().isoformat(),
                    "keywords": matched,
                    "content_snippet": article.text[:100],
                    "url_hash": generate_hash(url),
                    "full_content": article.text,
                    "matched_tags": matched,
                    "status": "New",
                    "method": "html"
                })
                logger.debug(f"  ✓ HTML: {article.title[:40] if article.title else 'Unknown'}...")
                
            except Exception as e:
                logger.debug(f"  HTML parse error: {str(e)[:30]}")
                continue
        
        if articles:
            return articles, "html"
    
    except Exception as e:
        logger.debug(f"{source_name}: HTML error - {str(e)[:40]}")
    
    return [], "none"


print("\n" + "=" * 70)
print("🚀 STARTING SCRAPER PIPELINE (CATEGORIZED SOURCES)")
print("=" * 70)

start_time = time.time()
logger.info("Starting scraper pipeline...")

all_articles = []

# Group sources by difficulty
easy_sources = [s for s in SOURCES_CONFIG if s['difficulty'] == 'easy']
medium_sources = [s for s in SOURCES_CONFIG if s['difficulty'] == 'medium']
hard_sources = [s for s in SOURCES_CONFIG if s['difficulty'] == 'hard']

# Phase 1: Smart scraping with fallback
print("\n📡 PHASE 1: SMART SCRAPING (RSS → HTML with timeouts)")
print("-" * 70)

for difficulty, sources in [("EASY", easy_sources), ("MEDIUM", medium_sources), ("HARD", hard_sources)]:
    print(f"\n   {difficulty} SOURCES ({len(sources)})")
    print("   " + "-" * 66)
    
    for source in sources:
        source_name = source['domain_name']
        language = source.get('language', 'Unknown')
        source_type = source.get('type', 'unknown')
        
        # Show progress
        print(f"   🔴 {source_name:<22} [{language:>5} | {source_type:>8}] ", end="", flush=True)
        source_start = time.time()
        
        articles, method = scrape_with_smart_fallback(source, KEYWORDS_CONFIG)
        elapsed = time.time() - source_start
        
        if articles:
            if method == "rss":
                print(f"📡 RSS  ✅ {len(articles):>2} articles] {elapsed:.1f}s")
            elif method == "html":
                print(f"🌐 HTML ✅ {len(articles):>2} articles] {elapsed:.1f}s")
            logger.info(f"{source_name}: Found {len(articles)} articles via {method.upper()} ({elapsed:.1f}s)")
            all_articles.extend(articles)
        else:
            if method == "skip":
                print(f"⏭️  SKIP (slow)] {elapsed:.1f}s")
            else:
                print(f"❌ NONE  (RSS/HTML failed)] {elapsed:.1f}s")
            logger.info(f"{source_name}: No articles - {method} ({elapsed:.1f}s)")

print("-" * 70)
print(f"📊 Total articles collected: {len(all_articles)}")

# Phase 2: Clean & Deduplicate
print("\n🧹 PHASE 2: CLEANING & DEDUPLICATION")
print("-" * 70)

cleaned_articles = clean_and_standardize(all_articles)
print(f"   Unique: {len(cleaned_articles)} articles")
print(f"   Duplicates removed: {len(all_articles) - len(cleaned_articles)}")

# Phase 3: Export to CSV
print("\n💾 PHASE 3: EXPORT TO CSV")
print("-" * 70)

exported_count = export_to_csv(cleaned_articles, OUTPUT_FILENAME)

# Summary with categorization
elapsed_time = time.time() - start_time
print("\n" + "=" * 70)
print("📊 FINAL SUMMARY")
print("=" * 70)

# Count by method
rss_count = sum(1 for a in cleaned_articles if a.get('method') == 'rss')
html_count = sum(1 for a in cleaned_articles if a.get('method') == 'html')

# Count by difficulty and language
easy_count = sum(1 for s in easy_sources for a in cleaned_articles if a['source'] == s['domain_name'])
medium_count = sum(1 for s in medium_sources for a in cleaned_articles if a['source'] == s['domain_name'])
hard_count = sum(1 for s in hard_sources for a in cleaned_articles if a['source'] == s['domain_name'])

english_count = sum(1 for s in SOURCES_CONFIG if s.get('language') == 'English' 
                    for a in cleaned_articles if a['source'] == s['domain_name'])
thai_count = sum(1 for s in SOURCES_CONFIG if s.get('language') == 'Thai' 
                 for a in cleaned_articles if a['source'] == s['domain_name'])

# Count by topic
tech_count = sum(1 for s in SOURCES_CONFIG if s.get('type') == 'tech' 
                 for a in cleaned_articles if a['source'] == s['domain_name'])
business_count = sum(1 for s in SOURCES_CONFIG if s.get('type') == 'business' 
                     for a in cleaned_articles if a['source'] == s['domain_name'])
general_count = sum(1 for s in SOURCES_CONFIG if s.get('type') == 'general' 
                    for a in cleaned_articles if a['source'] == s['domain_name'])

print(f"\n   📊 SOURCES:")
print(f"      Total: {len(SOURCES_CONFIG)} sources")
print(f"      Easy: {len(easy_sources)}, Medium: {len(medium_sources)}, Hard: {len(hard_sources)}")
print(f"\n   🌍 LANGUAGE DISTRIBUTION:")
print(f"      English: {english_count} articles")
print(f"      Thai: {thai_count} articles")
print(f"\n   📚 TOPIC DISTRIBUTION:")
print(f"      Tech: {tech_count} articles")
print(f"      Business: {business_count} articles")
print(f"      General: {general_count} articles")
print(f"\n   ⚙️  SCRAPING METHOD:")
print(f"      RSS: {rss_count} articles")
print(f"      HTML: {html_count} articles")
print(f"\n   📈 TOTAL UNIQUE ARTICLES: {exported_count}")
print(f"      Time elapsed: {elapsed_time:.1f}s")
print(f"      Output file: {OUTPUT_FILENAME}")
print("=" * 70)

logger.info(f"Pipeline complete in {elapsed_time:.1f}s")
logger.info(f"Final: {rss_count} RSS + {html_count} HTML = {exported_count} total from {len(SOURCES_CONFIG)} sources")


[INFO] Starting scraper pipeline...
[INFO] Starting scraper pipeline...
[INFO] Starting scraper pipeline...
[INFO] Starting scraper pipeline...
[INFO] Starting scraper pipeline...
[INFO] Starting scraper pipeline...
[INFO] Starting scraper pipeline...
[INFO] Starting scraper pipeline...
[INFO] Starting scraper pipeline...



🚀 STARTING SCRAPER PIPELINE (CATEGORIZED SOURCES)

📡 PHASE 1: SMART SCRAPING (RSS → HTML with timeouts)
----------------------------------------------------------------------

   EASY SOURCES (8)
   ------------------------------------------------------------------
   🔴 TechCrunch             [English |     tech] 

[INFO] TechCrunch: Found 20 articles via RSS (8.8s)
[INFO] TechCrunch: Found 20 articles via RSS (8.8s)
[INFO] TechCrunch: Found 20 articles via RSS (8.8s)
[INFO] TechCrunch: Found 20 articles via RSS (8.8s)
[INFO] TechCrunch: Found 20 articles via RSS (8.8s)
[INFO] TechCrunch: Found 20 articles via RSS (8.8s)
[INFO] TechCrunch: Found 20 articles via RSS (8.8s)
[INFO] TechCrunch: Found 20 articles via RSS (8.8s)
[INFO] TechCrunch: Found 20 articles via RSS (8.8s)


📡 RSS  ✅ 20 articles] 8.8s
   🔴 The Verge              [English |     tech] 

[INFO] The Verge: Found 9 articles via RSS (4.6s)
[INFO] The Verge: Found 9 articles via RSS (4.6s)
[INFO] The Verge: Found 9 articles via RSS (4.6s)
[INFO] The Verge: Found 9 articles via RSS (4.6s)
[INFO] The Verge: Found 9 articles via RSS (4.6s)
[INFO] The Verge: Found 9 articles via RSS (4.6s)
[INFO] The Verge: Found 9 articles via RSS (4.6s)
[INFO] The Verge: Found 9 articles via RSS (4.6s)
[INFO] The Verge: Found 9 articles via RSS (4.6s)


...

⏭️  SKIP (slow)] 11.4s
   🔴 Mashable               [English |     tech] 

[INFO] Mashable: Found 25 articles via RSS (28.5s)
[INFO] Mashable: Found 25 articles via RSS (28.5s)
[INFO] Mashable: Found 25 articles via RSS (28.5s)
[INFO] Mashable: Found 25 articles via RSS (28.5s)
[INFO] Mashable: Found 25 articles via RSS (28.5s)
[INFO] Mashable: Found 25 articles via RSS (28.5s)
[INFO] Mashable: Found 25 articles via RSS (28.5s)
[INFO] Mashable: Found 25 articles via RSS (28.5s)
[INFO] Mashable: Found 25 articles via RSS (28.5s)


📡 RSS  ✅ 25 articles] 28.5s
   🔴 Matichon               [ Thai |  general] 

[INFO] Matichon: No articles - none (11.8s)
[INFO] Matichon: No articles - none (11.8s)
[INFO] Matichon: No articles - none (11.8s)
[INFO] Matichon: No articles - none (11.8s)
[INFO] Matichon: No articles - none (11.8s)
[INFO] Cleaning 191 articles...
[INFO] Cleaning 191 articles...
[INFO] Cleaning 191 articles...
[INFO] Cleaning 191 articles...
[INFO] Matichon: No articles - none (11.8s)
[INFO] Matichon: No articles - none (11.8s)
[INFO] Matichon: No articles - none (11.8s)
[INFO] Matichon: No articles - none (11.8s)
[INFO] Cleaning 191 articles...
[INFO] Cleaning 191 articles...
[INFO] Cleaning 191 articles...
[INFO] Cleaning 191 articles...
[INFO] Cleaning 191 articles...
[INFO] Clean complete: 191 unique, 0 duplicates removed
[INFO] Clean complete: 191 unique, 0 duplicates removed
[INFO] Clean complete: 191 unique, 0 duplicates removed
[INFO] Clean complete: 191 unique, 0 duplicates removed
[INFO] Clean complete: 191 unique, 0 duplicates removed
[INFO] Exporting to scraped_data.csv

❌ NONE  (RSS/HTML failed)] 11.8s
----------------------------------------------------------------------
📊 Total articles collected: 191

🧹 PHASE 2: CLEANING & DEDUPLICATION
----------------------------------------------------------------------
   Unique: 191 articles
   Duplicates removed: 0

💾 PHASE 3: EXPORT TO CSV
----------------------------------------------------------------------

📊 FINAL SUMMARY

   📊 SOURCES:
      Total: 25 sources
      Easy: 8, Medium: 12, Hard: 5

   🌍 LANGUAGE DISTRIBUTION:
      English: 190 articles
      Thai: 1 articles

   📚 TOPIC DISTRIBUTION:
      Tech: 140 articles
      Business: 0 articles
      General: 51 articles

   ⚙️  SCRAPING METHOD:
      RSS: 175 articles
      HTML: 16 articles

   📈 TOTAL UNIQUE ARTICLES: 191
      Time elapsed: 513.1s
      Output file: scraped_data.csv


In [58]:
# 10. VERIFY OUTPUT

print("\n📋 VERIFICATION")
print("=" * 70)

try:
    with open(OUTPUT_FILENAME, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        
        print(f"✅ CSV file created successfully")
        print(f"   File: {OUTPUT_FILENAME}")
        print(f"   Rows: {len(rows)}")
        
        if rows:
            first_row = rows[0]
            print(f"\n📄 Sample Article:")
            print(f"   Source:     {first_row.get('source', 'N/A')}")
            print(f"   Headline:   {first_row.get('headline', 'N/A')[:60]}...")
            print(f"   Published:  {first_row.get('published', 'N/A')}")
            print(f"   Method:     {first_row.get('method', 'N/A')}")
            print(f"   Keywords:   {first_row.get('keywords', 'N/A')}")
            
except Exception as e:
    print(f"❌ Error: {e}")

print(f"\n📝 Logs: scraper_execution.log")
print("=" * 70)


📋 VERIFICATION
✅ CSV file created successfully
   File: scraped_data.csv
   Rows: 191

📄 Sample Article:
   Source:     TechCrunch
   Headline:   Rivian is building its own AI assistant...
   Published:  2025-12-10 04:00:10
   Method:     rss
   Keywords:   AI, Google, Microsoft, Meta, Technology, Model, OpenAI, AI

📝 Logs: scraper_execution.log


In [4]:
print("\n" + "=" * 70)
print("📰 SAMPLE NEWS ARTICLES (Mixed RSS & HTML)")
print("=" * 70)

try:
    # Read CSV and separate by method
    with open(OUTPUT_FILENAME, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        articles_list = list(reader)
    
    # Separate by method
    rss_articles = [a for a in articles_list if a.get('method') == 'rss']
    html_articles = [a for a in articles_list if a.get('method') == 'html']
    
    # Pick samples (mix of RSS and HTML)
    samples = []
    
    # Add 3 from RSS
    if rss_articles:
        samples.extend(rss_articles[:3])
    
    # Add 2 from HTML
    if html_articles:
        samples.extend(html_articles[:2])
    
    # Limit to 5 total
    samples = samples[:5]
    
    print(f"\nShowing {len(samples)} articles: {sum(1 for s in samples if s.get('method') == 'rss')} RSS + {sum(1 for s in samples if s.get('method') == 'html')} HTML\n")
    
    # Display each sample
    for idx, article in enumerate(samples, 1):
        method = article.get('method', 'unknown').upper()
        method_icon = "📡" if method == "RSS" else "🌐"
        
        print(f"{idx}. [{method_icon} {method}] {article.get('source', 'Unknown')}")
        print(f"   📌 Headline: {article.get('headline', 'N/A')}")
        print(f"   👤 Author: {article.get('author', 'Unknown')}")
        print(f"   📅 Published: {article.get('published', 'N/A')[:10]}")
        print(f"   🔑 Keywords: {article.get('keywords', 'N/A')}")
        print(f"   📄 Preview: {article.get('content_snippet', 'N/A')}")
        print()
    
    print("=" * 70)

except Exception as e:
    print(f"❌ Error displaying samples: {e}")



📰 SAMPLE NEWS ARTICLES (Mixed RSS & HTML)
❌ Error displaying samples: name 'OUTPUT_FILENAME' is not defined
